# 模型调用-invoke
1. 初始化模型

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="qwen2.5:3b-instruct-q4_K_M",
    model_provider="ollama",
    temperature=0.7,
    timeout=30,
    max_tokens=1000,
    max_retries=6,
)

e:\code\LangChainDemo\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


## 单次调用 invoke
调用模型的最直接方法是使用invoke()单个消息或消息列表。

In [2]:
response = model.invoke("Why do parrots have colorful feathers?")
print(response)

content="Parrots have colorful feathers for several important reasons related to their biology and behavior:\n\n1. **Attracting Mates**: One of the primary functions of bright colors in parrots is to attract mates. The vibrant plumage can be a signal of good health, genetic quality, and overall fitness.\n\n2. **Social Communication**: Colorful feathers help parrots communicate with other members of their species. Different patterns and colors can indicate things like age, sex, or social status within the group.\n\n3. **Camouflage in Their Natural Environment**: In many cases, the bright colors of a parrot's plumage are not for attracting mates but rather to blend into their environment. This is known as countershading, where darker coloring on the back and lighter coloring on the belly helps them hide from predators or prey.\n\n4. **Protection**: Bright feathers can also serve as a form of protection against certain threats. For example, some species use their colorful plumage to start

1. 聊天模型可以接受**消息列表**表示历史回话。
2. 每条消息都有一个角色，模型使用该角色来指示对话中消息的发送者。

In [3]:
conversation = [
    {"role": "system", "content": "You are a helpful assistant that translates English to French."},
    {"role": "user", "content": "Translate: I love programming."},
    {"role": "assistant", "content": "J'adore la programmation."},
    {"role": "user", "content": "Translate: I love building applications."}
]

response = model.invoke(conversation)
print(response)  # AIMessage("J'adore créer des applications.")

content='Je love construire des applications. \n\nNote: The phrase "Je love" is not a standard French expression. A more natural and correct translation would be "Je adore créer des applications."' additional_kwargs={} response_metadata={'model': 'qwen2.5:3b-instruct-q4_K_M', 'created_at': '2026-04-08T02:32:44.02989Z', 'done': True, 'done_reason': 'stop', 'total_duration': 918273800, 'load_duration': 126161200, 'prompt_eval_count': 55, 'prompt_eval_duration': 78428900, 'eval_count': 39, 'eval_duration': 626103900, 'logprobs': None, 'model_name': 'qwen2.5:3b-instruct-q4_K_M', 'model_provider': 'ollama'} id='lc_run--019d6aef-5564-72d2-9a84-6c5ce412c8e4-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 55, 'output_tokens': 39, 'total_tokens': 94}


In [4]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

conversation = [
    SystemMessage("You are a helpful assistant that translates English to French."),
    HumanMessage("Translate: I love programming."),
    AIMessage("J'adore la programmation."),
    HumanMessage("Translate: I love building applications.")
]

response = model.invoke(conversation)
print(response)  # AIMessage("J'adore créer des applications.")

content='Je love construire des applications. \n\nNote: The phrase "Je love" is not a standard French expression. A more natural and correct translation would be "Je adore créer des applications."' additional_kwargs={} response_metadata={'model': 'qwen2.5:3b-instruct-q4_K_M', 'created_at': '2026-04-08T02:33:07.2729958Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1039259200, 'load_duration': 126931200, 'prompt_eval_count': 55, 'prompt_eval_duration': 184884600, 'eval_count': 39, 'eval_duration': 640682200, 'logprobs': None, 'model_name': 'qwen2.5:3b-instruct-q4_K_M', 'model_provider': 'ollama'} id='lc_run--019d6aef-afb8-7b02-847f-3414a1768799-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 55, 'output_tokens': 39, 'total_tokens': 94}


## 流式调用 stream
大多数模型都能在生成输出内容的同时进行流式传输。通过逐步显示输出，流式传输显著提升了用户体验，尤其是在处理较长的响应时间。
调用stream()返回一个迭代器它会在生成过程中实时输出数据块。
可以使用循环来实时处理每个数据块：

In [5]:
for chunk in model.stream("Why do parrots have colorful feathers?"):
    print(chunk.text, end="|", flush=True)

Par|rots| have| colorful| feathers| for| several| important| reasons| related| to| their| biology| and| behavior|:

|1|.| **|At|tract|ing| M|ates|**:| One| of| the| primary| functions| of| bright| colors| in| par|rots| is| to| attract| mates|.| The| vibrant| plum|age| can| be| a| signal| of| good| health|,| genetic| quality|,| and| overall| fitness|.

|2|.| **|Social| Communication|**:| Color|ful| feathers| help| par|rots| communicate| with| other| members| of| their| species|.| Different| patterns| and| colors| can| indicate| things| like| age|,| sex|,| or| social| status| within| the| group|.

|3|.| **|Cam|ouflage| in| Their| Natural| Environment|**:| In| many| cases|,| the| bright| colors| of| a| par|rot|'s| plum|age| are| not| for| attracting| mates| but| rather| to| blend| into| their| environment|.| This| is| known| as| counters|h|ading|,| where| darker| coloring| on| the| back| and| lighter| coloring| on| the| belly| helps| them| hide| from| predators| or| prey|.

|4|.| **|Prote

与在模型生成完整响应后invoke()返回单个对象的函数不同，该函数返回多个对象，每个对象包含输出文本的一部分。
流中的每个数据块都旨在通过求和的方式组合成完整的消息：
AIMessagestream() AIMessageChunk

In [7]:
full = None  # None | AIMessageChunk
for chunk in model.stream("What color is the sky?"):
    full = chunk if full is None else full + chunk
    print(full.text)

# The
# The sky
# The sky is
# The sky is typically
# The sky is typically blue
# ...

full_content_blocks = getattr(full, 'content_blocks', None)
if full_content_blocks is not None:
    print(full_content_blocks)
    
# [{"type": "text", "text": "The sky is typically blue..."}]

The
The color
The color of
The color of the
The color of the sky
The color of the sky can
The color of the sky can vary
The color of the sky can vary depending
The color of the sky can vary depending on
The color of the sky can vary depending on the
The color of the sky can vary depending on the time
The color of the sky can vary depending on the time of
The color of the sky can vary depending on the time of day
The color of the sky can vary depending on the time of day and
The color of the sky can vary depending on the time of day and weather
The color of the sky can vary depending on the time of day and weather conditions
The color of the sky can vary depending on the time of day and weather conditions:


The color of the sky can vary depending on the time of day and weather conditions:

-
The color of the sky can vary depending on the time of day and weather conditions:

- During
The color of the sky can vary depending on the time of day and weather conditions:

- During the
The col

## 批次调用 batch
将一系列独立的模型请求批量处理，可以显著提高性能并降低成本，因为可以并行处理这些请求。
默认情况下，batch()只会返回整个批次的最终输出。

In [ ]:
responses = model.batch([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?"
])

for response in responses:
    print(response)

content="Parrots have colorful feathers for several important reasons related to their biology and behavior:\n\n1. **Attracting Mates**: One of the primary functions of bright colors in parrots is to attract mates. The vibrant plumage can be a signal of good health, genetic quality, and overall fitness.\n\n2. **Social Communication**: Colorful feathers also play a role in social communication among parrots. They use these colors during courtship displays or as part of their interactions with other members of the flock.\n\n3. **Camouflage**: In some environments, bright colors can actually serve as camouflage. For example, certain species of parrots may have patterns that help them blend into their surroundings when perched in trees or on branches.\n\n4. **Protection**: Bright feathers might also provide protection from predators by making it easier for the bird to be seen and avoided.\n\n5. **Thermoregulation**: The colorful parts of a parrot's plumage can play a role in thermoregulati

### batch_as_completed
如果您希望在每个输入生成完成后立即接收其输出，可以使用以下方式流式传输结果batch_as_completed()。
使用时batch_as_completed()，结果可能乱序。每个结果都包含用于匹配的输入索引，以便根据需要重建原始顺序。

In [8]:
for response in model.batch_as_completed([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?"
]):
    print(response)

(0, AIMessage(content="Parrots have colorful feathers for several important reasons related to their biology and behavior:\n\n1. **Attracting Mates**: One of the primary functions of bright colors in parrots is to attract mates. The vibrant plumage can be a signal of good health, genetic quality, and overall fitness.\n\n2. **Social Communication**: Colorful feathers also play a role in social communication among parrots. They use these colors during courtship displays or as part of their interactions with other members of the flock.\n\n3. **Camouflage**: In some environments, bright colors can actually serve as camouflage. For example, certain species of parrots may have patterns that help them blend into their surroundings when perched in trees or on branches.\n\n4. **Protection**: Bright feathers might also provide protection from predators by making it easier for the bird to be seen and avoided.\n\n5. **Thermoregulation**: The colorful parts of a parrot's plumage can play a role in 

### 控制并发数目

In [ ]:
from langchain_core.language_models import LanguageModelInput

# 控制并发数
list_inputs: list[LanguageModelInput] = [
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?",
]

responses = model.batch(
    list_inputs,
    config={
        'max_concurrency': 2,
    }
)

for response in responses:
    print(response)

content="Parrots have colorful feathers for several important reasons related to their biology and behavior:\n\n1. **Attracting Mates**: One of the primary functions of bright colors in parrots is to attract mates. The vibrant plumage can be a signal of good health, genetic quality, and overall fitness.\n\n2. **Social Communication**: Colorful feathers help parrots communicate with other members of their species. Different patterns and colors can indicate things like age, sex, or social status within the group.\n\n3. **Camouflage in Their Natural Environment**: In many cases, the bright colors of a parrot's plumage are not for attracting mates but rather to blend into their environment. This is known as countershading, where darker coloring on the back and lighter coloring on the belly helps them hide from predators or prey.\n\n4. **Protection**: Bright feathers can also serve as a form of protection against certain threats. For example, some species use their colorful plumage to start

✅ 模型开始运行
✅ 模型运行结束

👇 回答：
Sure! Here's one for you:

Why couldn't the bicycle stand up by itself?

Because it was leaning towards "bicycle" (won't)!

I hope you found that amusing! Do you have any preference for another type of joke?
